In [1]:
import sqlite3
import numpy as np
import pandas as pd
conn = sqlite3.connect('database.sqlite')

In [2]:
def standings(league, season):
    
    #creating dataframe from our sql db
    matches_df = pd.read_sql(
    "SELECT league_id, season, match_api_id, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal FROM match",
    conn, index_col = "match_api_id")
    
    #creating column with match results
    conditions = [
        (matches_df["home_team_goal"]>matches_df["away_team_goal"]),
        (matches_df["home_team_goal"]<matches_df["away_team_goal"]),
        (matches_df["home_team_goal"]==matches_df["away_team_goal"])
    ]
    choices = ['home_win', 'away_win', 'draw']
    matches_df['result'] = np.select(conditions, choices, default='Unknown')

    #adding league names to dataframe matches_df
    leagues_df = pd.read_sql('SELECT id,name FROM league', conn)
    matches_df = matches_df.merge(leagues_df, left_on = "league_id", right_on = "id")
    matches_df = matches_df.drop("id", axis = 1)
    matches_df = matches_df.drop("league_id", axis = 1)

    #checking if data is correct
    season = season.replace(" ", "/").replace("-", "/")
    season_exists = (matches_df["season"] == season).any()
    league_exists = matches_df["name"].str.contains(
        league,
        case = False,
        na = False
    ).any()
    
    if not season_exists and not league_exists:
        return "Season and league not found!"
    elif not season_exists:
        return "Season not found!"
    elif not league_exists:
        return "League not found!"
    else:
        league_matches =  matches_df[(
            matches_df['season'] == season) & 
            (matches_df['name'].str.contains(league, case=False, na=False))
            ].copy()

    #checking if data is found
    if league_matches.empty:
        return "No matches found. Check league name or season."
    
    #creating column with points at home
    conditions = [
        (league_matches["result"] == "home_win"),
        (league_matches["result"] == "draw"),
        (league_matches["result"] == "away_win")
    ]
    choices = [3, 1, 0]
    league_matches['home_team_pts'] = np.select(conditions, choices, default=0)

    #creating column with points away
    conditions = [
        (league_matches["result"] == "home_win"),
        (league_matches["result"] == "draw"),
        (league_matches["result"] == "away_win")
    ]
    choices = [0, 1, 3]
    league_matches['away_team_pts'] = np.select(conditions, choices, default=0)
    league_name = league_matches["name"].iloc[0]
    
    #counting home, away, then total points and creating corresponding columns
    home_points = league_matches.groupby("home_team_api_id")["home_team_pts"].sum()
    away_points = league_matches.groupby("away_team_api_id")["away_team_pts"].sum()
    total_points = away_points + home_points
    total_points.name = "Pts"
    
    #counting home, away, then total goals scored and creating corresponding columns
    home_goals_scored = league_matches.groupby("home_team_api_id")["home_team_goal"].sum()
    away_goals_scored = league_matches.groupby("away_team_api_id")["away_team_goal"].sum()
    total_goals_scored = home_goals_scored + away_goals_scored    
    total_goals_scored.name = "GF"
    
    #counting home, away, then total goals conceded and creating corresponding columns
    home_goals_conceded = league_matches.groupby("home_team_api_id")["away_team_goal"].sum()
    away_goals_conceded = league_matches.groupby("away_team_api_id")["home_team_goal"].sum()
    total_goals_conceded = home_goals_conceded + away_goals_conceded
    total_goals_conceded.name = "GA"
    
    #creating dataframes with wins, loses and draws
    home_record = league_matches.groupby("home_team_api_id")["result"].value_counts().unstack().fillna(0).astype(int)
    away_record = league_matches.groupby("away_team_api_id")["result"].value_counts().unstack().fillna(0).astype(int)

    #creating a table with awailable data
    standings_table = pd.concat([total_points, total_goals_scored, total_goals_conceded], axis=1) 
    standings_table["W"] = home_record["home_win"] + away_record["away_win"]
    standings_table["D"] = home_record["draw"] + away_record["draw"]
    standings_table["L"] = home_record["away_win"] + away_record["home_win"]
    standings_table["GD"] = standings_table["GF"] - standings_table["GA"]
    standings_table['MP'] = standings_table["W"] + standings_table["D"] + standings_table["L"] 
    
    #adding club names to standings_table
    teams_df = pd.read_sql(
    "SELECT team_api_id, team_long_name FROM team", conn, index_col = "team_api_id")
    standings_table["Club"] = standings_table.index.map(teams_df["team_long_name"])
    
    #changing the order of columns
    standings_table = standings_table[
    [
        "Club",
        "MP",
        "W",
        "D",
        "L",
        "GF",
        "GA",
        "GD",
        "Pts"
    ]]
    standings_table = standings_table.sort_values(
    ["Pts", "GD", "GF"],
    ascending=[False, False, False])

    #changing the range so the first team would be first, not 0
    standings_table.index = range(1, len(standings_table) + 1)

    return (standings_table.style \
    .set_caption(f"{league_name} {season}") \
    .set_properties(**{"text-align": "center"}))
    
    

In [3]:
standings("Poland","2015 2016")

,Club,MP,W,D,L,GF,GA,GD,Pts
1,Legia Warszawa,30,17,9,4,58,28,30,60
2,Piast Gliwice,30,17,7,6,49,36,13,58
3,Pogoń Szczecin,30,10,16,4,36,30,6,46
4,Cracovia,30,12,9,9,57,42,15,45
5,Zagłębie Lubin,30,12,9,9,41,37,4,45
6,Lech Poznań,30,13,4,13,37,38,-1,43
7,Lechia Gdańsk,30,10,9,11,45,37,8,39
8,Ruch Chorzów,30,11,6,13,37,46,-9,39
9,Podbeskidzie Bielsko-Biała,30,9,11,10,37,46,-9,38
10,Wisła Kraków,30,8,13,9,45,35,10,37
